# Recipe Ingredient Predictor using Bidirectional RNN



In [ ]:
#Install Libraries
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense
from tensorflow.keras.utils import to_categorical

In [ ]:
#Small Dataset
recipes = [
    "onion tomato garlic",
    "rice chicken pepper",
    "bread butter jam",
    "milk sugar coffee",
    "pasta tomato cheese",
    "potato onion oil"
]

In [ ]:
#Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts(recipes)

total_words = len(tokenizer.word_index) + 1

In [ ]:
#create Sequences
input_sequences = []

for line in recipes:
    token_list = tokenizer.texts_to_sequences([line])[0]

    for i in range(1, len(token_list)):
        n_gram = token_list[:i+1]
        input_sequences.append(n_gram)

In [ ]:
#Padding
max_len = max(len(x) for x in input_sequences)

input_sequences = pad_sequences(input_sequences,
                                maxlen=max_len,
                                padding='pre')

In [ ]:
#Split X and Y
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

y = to_categorical(y, num_classes=total_words)

In [ ]:
#Build BiRNN Model
model = Sequential([
    Embedding(total_words, 10, input_length=max_len-1),

    Bidirectional(LSTM(16)),

    Dense(total_words, activation='softmax')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
#Compile and Train
model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

model.fit(X, y, epochs=200, verbose=0)

In [ ]:
#Predict Ingredient
text = "onion tomato"

token_list = tokenizer.texts_to_sequences([text])[0]

token_list = pad_sequences(
    [token_list],
    maxlen=max_len-1,
    padding='pre'
)

# Predict probabilities
pred = model.predict(token_list)[0]

# Get top 4 predictions
top_indices = pred.argsort()[-4:][::-1]

print("Top Predictions:")

for idx in top_indices:
    for word, index in tokenizer.word_index.items():
        if index == idx:
            print(word)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
Top Predictions:
garlic
cheese
tomato
pepper
